# Attention Rollout do SegFormer (estágio 14)

Gera mapas de relevância por **Attention Rollout** para o **SegFormer (ViT)** usando os pesos treinados no estágio 10 e os artefatos de avaliação do estágio 11 da **fonte escolhida manualmente** (`ground_truth.chosen_source`). Para patches representativos da validação espacial, o rollout é calculado na última etapa do codificador hierárquico, a única com atenção quadrada (redução espacial 1:1), com média das cabeças, conexões residuais e composição das camadas. Como o SegFormer não possui token CLS, a relevância de cada token é a média das linhas do rollout. Os mapas são persistidos em `MyDrive/tcc/artifacts/xai/attention_rollout/segformer/<fonte>/` e visualizados em uma figura com RGB, referência, mapa e sobreposição. O estágio não retreina: apenas explica os pesos existentes, usando o modelo da dobra de cada patch.

## Bootstrap do workspace

O primeiro passo baixa e executa `src/bootstrap.py` (somente stdlib) — necessário porque o `src/` ainda não está disponível para import em uma sessão nova. O bootstrap obtém o repositório público, extrai `src/`, `data/external/` e `requirements-runtime.txt` para o workspace e adiciona o workspace ao `sys.path`. O `reload` garante que reexecuções usem a versão mais recente baixada.

In [ ]:
# Baixa e executa o bootstrap do workspace (etapa prévia ao import de src/).
import importlib
import pathlib
import sys
import urllib.request

BOOTSTRAP_URL = "https://raw.githubusercontent.com/jotap1101/tcc-lab/main/src/bootstrap.py"
pathlib.Path("bootstrap.py").write_bytes(urllib.request.urlopen(BOOTSTRAP_URL).read())
sys.path.insert(0, str(pathlib.Path.cwd()))

# Recarrega o módulo para não reutilizar uma versão antiga em cache no kernel.
bootstrap = importlib.import_module("bootstrap")
importlib.reload(bootstrap)

workspace = bootstrap.bootstrap_workspace()
print(f"Workspace: {workspace}")

## Dependências pinadas

Garante as versões fixadas em `requirements-runtime.txt`, o mesmo conjunto nas duas plataformas. Quando uma biblioteca com extensão compilada já importada (numpy, scipy, torch, entre outras) diverge da versão pinada, o kernel é reiniciado automaticamente para evitar inconsistência de ABI; nesse caso, o notebook é reexecutado a partir da primeira célula. Como o estágio carrega o SegFormer via Transformers, também remove um `torchaudio` incompatível que impediria o import.


In [ ]:
# Garante as versões pinadas, reinicia o kernel em divergência de ABI e remove
# torchaudio incompatível que impediria o import do transformers.
from src.setup import install_runtime_requirements, remove_incompatible_torchaudio

requirements = pathlib.Path(workspace) / "requirements-runtime.txt"
install_runtime_requirements(requirements)
remove_incompatible_torchaudio()


## Pacote compartilhado, plataforma e armazenamento

Importa o pacote `src/` (já entregue pelo bootstrap) e identifica a plataforma pela abstração em `src/io.py`. Em seguida garante a raiz `MyDrive/tcc/` e resolve todos os caminhos de armazenamento definidos em `src/config.yaml`, incluindo modelos, artefatos de avaliação, mapas de XAI do SegFormer e figuras. A fonte explicada é a mesma escolhida manualmente na fase de ground truth.

In [ ]:
# Resolve o contexto de runtime: plataforma, armazenamento canônico e configuração.
from src.data.source_decision import chosen_source
from src.setup import initialize
from src.xai.attention_rollout import (
    attention_rollout_dir,
    attention_rollout_figure_path,
    attention_rollout_maps_path,
)

ctx = initialize(workspace)
platform = ctx.platform
storage_paths = ctx.storage_paths
config = ctx.config
source_name = chosen_source()
print(f"Plataforma: {platform}")
print(f"Fonte de treinamento/avaliação (decisão manual): {source_name}")
print(f"Mapas Attention Rollout: {attention_rollout_dir(storage_paths, source_name)}")
print(f"Arquivo de mapas: {attention_rollout_maps_path(storage_paths, source_name)}")
print(f"Figura: {attention_rollout_figure_path(storage_paths, source_name)}")

## Reprodutibilidade e dispositivo

Fixa as sementes de python/numpy/torch/cuda e habilita as flags determinísticas do PyTorch, além de registrar as versões do ambiente. O Attention Rollout apenas executa o forward do backbone com `output_attentions=True` em modo `eager`, sem atualizar pesos e sem backward; por isso as flags estritas permanecem ativas. O processamento ocorre na GPU quando disponível e recai para CPU quando não há acelerador — viável porque apenas poucos patches representativos são explicados.

In [ ]:
# Fixa sementes, flags determinísticas e seleciona o dispositivo de execução.
import torch

from src.utils import log_environment, setup_reproducibility

setup_reproducibility(config)
log_environment()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print(f"Dispositivo: {device} | {torch.cuda.get_device_name(0)}")
else:
    print(f"Dispositivo: {device} (GPU indisponível; Attention Rollout em CPU)")

## Dependências dos estágios 06, 08, 10 e 11

Verifica que o manifesto (estágio 06), as estatísticas de normalização (estágio 08), os pesos e artefatos de treino do SegFormer (estágio 10) e o resumo vigente da avaliação (estágio 11) estão disponíveis. A explicação usa o modelo da dobra de cada patch selecionado, sem reavaliar todos os patches.

In [ ]:
# Verifica as dependências dos estágios anteriores e a vigência dos artefatos do SegFormer.
from src.data.eda import normalization_stats_is_current, normalization_stats_path
from src.data.patch_generation import manifest_is_current, manifest_path
from src.data.spatial_split import fold_count, split_is_current
from src.evaluation import evaluation_summary_path, summary_is_current
from src.trainer import fold_training_done
from src.utils import check_dependencies

check_dependencies(
    {
        "Manifesto (estágio 06)": manifest_path(storage_paths, source_name),
        "Estatísticas de normalização (estágio 08)": normalization_stats_path(
            storage_paths, source_name
        ),
        "Resumo da avaliação (estágio 11)": evaluation_summary_path(
            storage_paths, "segformer", source_name
        ),
    }
)
if not manifest_is_current(storage_paths, source_name):
    raise ValueError("Manifesto desatualizado frente às entradas; execute o estágio 06 antes.")
if not split_is_current(storage_paths, source_name):
    raise ValueError("Divisão espacial desatualizada; execute o estágio 07 antes.")
if not normalization_stats_is_current(storage_paths, source_name):
    raise ValueError("Estatísticas de normalização desatualizadas; execute o estágio 08 antes.")
if not summary_is_current(storage_paths, "segformer", source_name):
    raise ValueError(
        "Resumo do SegFormer desatualizado; reexecute o estágio 11 antes do Attention Rollout."
    )
for fold in range(fold_count()):
    if not fold_training_done(storage_paths, "segformer", source_name, fold):
        raise FileNotFoundError(
            f"Pesos/artefatos de treino ausentes para o SegFormer, dobra {fold}; "
            "execute o estágio 10 antes do Attention Rollout."
        )
print(f"Dobras do SegFormer verificadas: {fold_count()}")

## Geração dos mapas Attention Rollout

Seleciona de forma determinística patches representativos com café (maior e menor IoU, maior proporção de café e IoU mediano), carrega a imagem normalizada e a máscara de referência, e calcula o rollout com os pesos da dobra de cada patch. O rollout usa a última etapa do codificador, agrega as cabeças pela média, soma a identidade (conexões residuais), normaliza cada linha e compõe as camadas; a relevância dos tokens é reamostrada bilinearmente para a resolução de entrada. Os mapas, o metadata com fingerprint e a figura são persistidos no Drive; a geração é reutilizada apenas enquanto o resumo do estágio 11, os pesos das dobras e a configuração permanecem vigentes.

In [ ]:
# Gera (ou reutiliza) os mapas Attention Rollout do SegFormer na fonte escolhida.
from src.xai.attention_rollout import run_attention_rollout

rollout_result = run_attention_rollout(storage_paths, source_name=source_name, device=device)
print(f"Artefatos reutilizados: {rollout_result.skipped}")
print(f"Patches explicados: {rollout_result.metadata['n_patches']}")

## Patches explicados e métricas

Apresenta os patches selecionados, a dobra que forneceu os pesos, o IoU do SegFormer, a proporção de café e o hash dos pesos usados em cada mapa, permitindo rastrear exatamente qual modelo gerou cada explicação.

In [ ]:
# Monta a tabela dos patches explicados e dos pesos utilizados.
import pandas as pd

pd.DataFrame(
    [
        {
            "patch_id": item["patch_id"],
            "caso": item["label"],
            "dobra": item["fold"],
            "iou": round(item["iou"], 4),
            "coffee_ratio": round(item["coffee_ratio"], 4),
            "weights_sha256": item["weights_sha256"],
        }
        for item in rollout_result.metadata["patches"]
    ]
)

## Resumo da etapa

Exibe o resumo do estágio com a fonte, o escopo e a etapa do rollout, a grade de tokens, o número de cabeças, o número de patches explicados e os caminhos dos mapas, do metadata e da figura.

In [ ]:
# Exibe o resumo da etapa e os caminhos dos artefatos de XAI.
from src.utils import print_summary

rollout_metadata = rollout_result.metadata["rollout"]
summary = {
    "Fonte": source_name,
    "Modelo": rollout_result.metadata["model"],
    "Escopo do rollout": rollout_metadata["rollout_scope"],
    "Etapa": rollout_metadata["stage_index"],
    "Grade de tokens": "x".join(str(value) for value in rollout_metadata["stage_grid"]),
    "Cabeças": rollout_metadata["num_heads"],
    "Patches explicados": rollout_result.metadata["n_patches"],
    "Dispositivo": rollout_result.metadata["device"],
    "Mapas": str(rollout_result.maps_path),
    "Metadata": str(rollout_result.metadata_path),
    "Figura": str(rollout_result.figure_path),
}
print_summary(summary, "14")